In [ ]:
!pip install anthropic -q

import anthropic
import pandas as pd
import json
import time
import random

try:
    from google.colab import files
    print('Upload your two CSV files:')
    print('  - bumble_combined.csv')
    print('  - bumble_training_set.csv')
    uploaded = files.upload()
    print(f'Uploaded: {list(uploaded.keys())}')
except ImportError:
    print('Not in Colab — reading from local directory.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.0/833.0 kB 12.2 MB/s eta 0:00:00
Upload your two CSV files:
  - bumble_combined.csv
  - bumble_training_set.csv


Saving bumble_combined_full.csv to bumble_combined_full.csv
Saving bumble_training_set.csv to bumble_training_set.csv
Uploaded: ['bumble_combined_full.csv', 'bumble_training_set.csv']


In [ ]:
from google.colab import userdata
API_KEY = userdata.get('bumble_API')
client = anthropic.Anthropic(api_key=API_KEY)

In [ ]:
from google.colab import userdata
import anthropic

# ---------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------
MODEL          = "claude-haiku-4-5-20251001"
BATCH_SIZE     = 100
FEW_SHOT_COUNT = 6
OUTPUT_FILE    = "bumble_classified.csv"

client = anthropic.Anthropic(api_key=userdata.get('bumble_API'))
print('Anthropic client ready.')
print(f'Model: {MODEL}')

Anthropic client ready.
Model: claude-haiku-4-5-20251001


In [ ]:
import os
print(os.listdir('.'))

['.config', 'bumble_training_set.csv', 'bumble_combined_full.csv', 'sample_data']


In [ ]:
# ---------------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------------
df_main     = pd.read_csv('bumble_combined_full.csv')
df_training = pd.read_csv('bumble_training_set.csv')

# Only use training rows that have a category label
df_training = df_training[df_training['category_1'].notna()].reset_index(drop=True)

print(f'Main dataset:    {len(df_main)} rows to classify')
print(f'Training set:    {len(df_training)} labelled examples')
print(f'\nTraining category distribution:')
print(df_training['category_1'].value_counts().head(10).to_string())

Main dataset:    56304 rows to classify
Training set:    283 labelled examples

Training category distribution:
category_1
Forced subscription     33
No matches              26
Bad quality matches     17
Account issues          14
Ghosting                14
Dating fatigue          14
Poor value for money    14
UX issues               12
Pricing issues          11
Hookup culture          11


In [ ]:
# ---------------------------------------------------------------
# VALID CATEGORIES AND DIMENSIONS
# ---------------------------------------------------------------
VALID_CATEGORIES = [
    # Dim 1
    'Good quality matches', 'Bad quality matches', 'Ghosting',
    'Low effort interactions', 'Hookup culture', 'Relationship success',
    'Relationship mismatch', 'Validation seeking', 'No matches', 'Too many options',
    # Dim 2
    'Dating fatigue', 'Hopelessness', 'Frustration',
    'Positive emotional experience', 'Insecurity',
    # Dim 3
    'Fake profiles & bots', 'Scams', 'Harassment & safety',
    'Verification', 'Good support', 'Poor support', 'Account issues',
    # Dim 4
    'Forced subscription', 'Poor value for money', 'Good value for money',
    'Pricing issues', 'Monetisation manipulation',
    # Dim 5
    'Better than competition', 'Worse than competition',
    'Women-first positive', 'Women-first negative',
    # Dim 6
    'UX issues', 'UX positive', 'Algorithm issues', 'Bugs',
    # Other
    'Uncategorised'
]

DIMENSION_MAP = {
    'Good quality matches':          'Dim 1 - Match Quality',
    'Bad quality matches':           'Dim 1 - Match Quality',
    'Ghosting':                      'Dim 1 - Match Quality',
    'Low effort interactions':       'Dim 1 - Match Quality',
    'Hookup culture':                'Dim 1 - Match Quality',
    'Relationship success':          'Dim 1 - Match Quality',
    'Relationship mismatch':         'Dim 1 - Match Quality',
    'Validation seeking':            'Dim 1 - Match Quality',
    'No matches':                    'Dim 1 - Match Quality',
    'Too many options':              'Dim 1 - Match Quality',
    'Dating fatigue':                'Dim 2 - Emotional Experience',
    'Hopelessness':                  'Dim 2 - Emotional Experience',
    'Frustration':                   'Dim 2 - Emotional Experience',
    'Positive emotional experience': 'Dim 2 - Emotional Experience',
    'Insecurity':                    'Dim 2 - Emotional Experience',
    'Fake profiles & bots':          'Dim 3 - Trust & Safety',
    'Scams':                         'Dim 3 - Trust & Safety',
    'Harassment & safety':           'Dim 3 - Trust & Safety',
    'Verification':                  'Dim 3 - Trust & Safety',
    'Good support':                  'Dim 3 - Trust & Safety',
    'Poor support':                  'Dim 3 - Trust & Safety',
    'Account issues':                'Dim 3 - Trust & Safety',
    'Forced subscription':           'Dim 4 - Monetisation',
    'Poor value for money':          'Dim 4 - Monetisation',
    'Good value for money':          'Dim 4 - Monetisation',
    'Pricing issues':                'Dim 4 - Monetisation',
    'Monetisation manipulation':     'Dim 4 - Monetisation',
    'Better than competition':       'Dim 5 - Brand & Competition',
    'Worse than competition':        'Dim 5 - Brand & Competition',
    'Women-first positive':          'Dim 5 - Brand & Competition',
    'Women-first negative':          'Dim 5 - Brand & Competition',
    'UX issues':                     'Dim 6 - Product & UX',
    'UX positive':                   'Dim 6 - Product & UX',
    'Algorithm issues':              'Dim 6 - Product & UX',
    'Bugs':                          'Dim 6 - Product & UX',
    'Uncategorised':                 'Uncategorised',
}

print(f'Valid categories: {len(VALID_CATEGORIES)}')

Valid categories: 36


In [ ]:
# ---------------------------------------------------------------
# BUILD PROMPT
# ---------------------------------------------------------------

SYSTEM_PROMPT = """You are a sentiment classification assistant for a consulting project analysing Bumble (the dating app).

Your task is to classify a comment or review into 1-3 categories from the list below.
Only use categories from this exact list. If the comment is not about the app or dating experience at all, use 'Uncategorised'.

VALID CATEGORIES:
Dim 1 - Match Quality: Good quality matches, Bad quality matches, Ghosting, Low effort interactions, Hookup culture, Relationship success, Relationship mismatch, Validation seeking, No matches, Too many options
Dim 2 - Emotional Experience: Dating fatigue, Hopelessness, Frustration, Positive emotional experience, Insecurity
Dim 3 - Trust & Safety: Fake profiles & bots, Scams, Harassment & safety, Verification, Good support, Poor support, Account issues
Dim 4 - Monetisation: Forced subscription, Poor value for money, Good value for money, Pricing issues, Monetisation manipulation
Dim 5 - Brand & Competition: Better than competition, Worse than competition, Women-first positive, Women-first negative
Dim 6 - Product & UX: UX issues, UX positive, Algorithm issues, Bugs
Other: Uncategorised

Respond ONLY with a JSON object in this exact format:
{"category_1": "...", "category_2": "..." or null, "category_3": "..." or null, "confidence": "high" or "medium" or "low"}"""

def build_few_shot_examples(n=6):
    """Sample n examples from training set, balanced across categories."""
    sample = df_training.groupby('category_1', group_keys=False).apply(
        lambda x: x.sample(min(1, len(x)), random_state=42)
    ).sample(min(n, len(df_training)), random_state=42)

    examples = []
    for _, row in sample.iterrows():
        cat2 = row['category_2'] if pd.notna(row.get('category_2')) else None
        cat3 = row['category_3'] if pd.notna(row.get('category_3')) else None
        result = {"category_1": row['category_1'], "category_2": cat2,
                  "category_3": cat3, "confidence": "high"}
        examples.append(f"Comment: {str(row['text'])[:300]}\nClassification: {json.dumps(result)}")
    return "\n\n".join(examples)

def build_user_message(text, few_shot_examples):
    return f"""Here are some example classifications:\n\n{few_shot_examples}\n\nNow classify this comment:\nComment: {str(text)[:500]}"""

print('Prompt functions ready.')

Prompt functions ready.


In [ ]:
# ---------------------------------------------------------------
# TEST ON 5 ROWS BEFORE RUNNING FULL BATCH
# ---------------------------------------------------------------
print('Testing on 5 rows...')
few_shot = build_few_shot_examples(6)
test_sample = df_main.sample(5, random_state=42)

for i, (_, row) in enumerate(test_sample.iterrows()):
    try:
        response = client.messages.create(
            model=MODEL,
            max_tokens=150,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": build_user_message(row['text'], few_shot)}]
        )
        raw = response.content[0].text.strip()
        raw = raw.replace("```json", "").replace("```", "").strip()
        if not raw:
            print(f"\nRow {i+1}: Empty response — skipping")
            continue
        result = json.loads(raw)
        print(f"\nRow {i+1}:")
        print(f"  Text: {str(row['text'])[:80]}...")
        print(f"  Classification: {result}")
    except json.JSONDecodeError:
        print(f"\nRow {i+1}: Could not parse response — raw output: '{response.content[0].text[:100]}'")
    except Exception as e:
        print(f"\nRow {i+1}: Error — {e}")

print('\nTest complete. If results look good, run the batch cell below.')

Testing on 5 rows...


NameError: name 'build_few_shot_examples' is not defined

In [ ]:
from google.colab import userdata

BATCH_ID = userdata.get('BATCH_ID')
print(f'Retrieving results for: {BATCH_ID}')

results = {}
for result in client.messages.batches.results(BATCH_ID):
    idx = int(result.custom_id)
    if result.result.type == 'succeeded':
        try:
            raw = result.result.message.content[0].text.strip()
            raw = raw.replace("```json", "").replace("```", "").strip()
            parsed = json.loads(raw)
            results[idx] = parsed
        except:
            results[idx] = {"category_1": "Uncategorised", "category_2": None,
                            "category_3": None, "confidence": "low"}
    else:
        results[idx] = {"category_1": "Uncategorised", "category_2": None,
                        "category_3": None, "confidence": "low"}

print(f'Retrieved {len(results)} results')

Retrieving results for: msgbatch_01MDvfwMcN8k53yqCX2GnxPp
Retrieved 56304 results


In [ ]:
# ---------------------------------------------------------------
# CHECK BATCH STATUS
# Run this cell periodically to check if batch is complete
# ---------------------------------------------------------------

# If session restarted, paste your Batch ID here:
# BATCH_ID = "msgbatch_xxxxxxxxxxxx"

batch_status = client.messages.batches.retrieve(BATCH_ID)
print(f'Batch ID:  {BATCH_ID}')
print(f'Status:    {batch_status.processing_status}')
print(f'Succeeded: {batch_status.request_counts.succeeded}')
print(f'Errored:   {batch_status.request_counts.errored}')
print(f'Pending:   {batch_status.request_counts.processing}')

if batch_status.processing_status == 'ended':
    print('\nBatch complete — run the next cell to retrieve results.')
else:
    print('\nNot yet complete — wait and re-run this cell.')

Batch ID:  msgbatch_01MDvfwMcN8k53yqCX2GnxPp
Status:    ended
Succeeded: 56299
Errored:   5
Pending:   0

Batch complete — run the next cell to retrieve results.


In [ ]:
# ---------------------------------------------------------------
# RETRIEVE RESULTS AND BUILD CLASSIFIED DATASET
# Only run once batch status shows 'ended'
# ---------------------------------------------------------------
print('Retrieving batch results...')

results = {}
for result in client.messages.batches.results(BATCH_ID):
    idx = int(result.custom_id)
    if result.result.type == 'succeeded':
        try:
            raw = result.result.message.content[0].text.strip()
            raw = raw.replace("```json", "").replace("```", "").strip()
            parsed = json.loads(raw)
            results[idx] = parsed
        except:
            results[idx] = {"category_1": "Uncategorised", "category_2": None,
                            "category_3": None, "confidence": "low"}
    else:
        results[idx] = {"category_1": "Uncategorised", "category_2": None,
                        "category_3": None, "confidence": "low"}

print(f'Retrieved {len(results)} results')

# Add classification columns to main dataframe
df_classified = df_main.copy()
df_classified['category_1']               = df_classified.index.map(lambda i: results.get(i, {}).get('category_1', 'Uncategorised'))
df_classified['category_2']               = df_classified.index.map(lambda i: results.get(i, {}).get('category_2'))
df_classified['category_3']               = df_classified.index.map(lambda i: results.get(i, {}).get('category_3'))
df_classified['classification_confidence']= df_classified.index.map(lambda i: results.get(i, {}).get('confidence', 'low'))

# Add dimension columns
df_classified['dimension_1'] = df_classified['category_1'].map(DIMENSION_MAP)
df_classified['dimension_2'] = df_classified['category_2'].map(DIMENSION_MAP)

print(f'\nClassification complete: {len(df_classified)} rows')
print(f'\nCategory distribution (top 15):')
print(df_classified['category_1'].value_counts().head(15).to_string())
print(f'\nDimension distribution:')
print(df_classified['dimension_1'].value_counts().to_string())
print(f'\nConfidence distribution:')
print(df_classified['classification_confidence'].value_counts().to_string())

Retrieving batch results...
Retrieved 56304 results

Classification complete: 56304 rows

Category distribution (top 15):
category_1
Uncategorised                    7024
Forced subscription              6740
Account issues                   5095
Positive emotional experience    3497
Low effort interactions          2781
No matches                       2742
UX issues                        2517
Fake profiles & bots             2379
Monetisation manipulation        2320
Bad quality matches              1969
Poor value for money             1908
Algorithm issues                 1762
Relationship mismatch            1501
Worse than competition           1459
Bugs                             1305

Dimension distribution:
dimension_1
Dim 4 - Monetisation            11858
Dim 1 - Match Quality           11739
Dim 3 - Trust & Safety          10092
Uncategorised                    7024
Dim 6 - Product & UX             6379
Dim 2 - Emotional Experience     5459
Dim 5 - Brand & Competition     

In [ ]:
df_classified.to_csv(OUTPUT_FILE, index=False)
print(f'Saved {len(df_classified)} rows to {OUTPUT_FILE}')
print(f'Columns: {list(df_classified.columns)}')

Saved 56304 rows to bumble_classified.csv
Columns: ['source', 'original_date', 'comment_date', 'text', 'comment', 'subreddit', 'post_title', 'post_score', 'comment_score', 'rating', 'thumbs_up', 'vader_compound', 'vader_positive', 'vader_negative', 'vader_neutral', 'sentiment_label', 'category_1', 'category_2', 'category_3', 'dimension_1', 'dimension_2', 'classification_confidence']


In [ ]:
try:
    from google.colab import files
    files.download(OUTPUT_FILE)
    print('Download triggered.')
except ImportError:
    print(f'Not in Colab — file saved locally as {OUTPUT_FILE}')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered.
